# Behind the scenes: Day Two
### Data download and preparation

1. Beck et al. ALREADY DONE IN DAY1_BTS!
2. GBIF species occurrence records (Philippine Eagle, Flying Fox) + a broader mammal community dataset
3. Sentinel-2 satellite imagery for Negros island
4. Hansen Global Forest Change data for Negros

In [ ]:
from pathlib import Path
import re
import requests
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
import rioxarray
from rasterio.windows import from_bounds
from rasterio.io import MemoryFile
from rasterio.merge import merge as rio_merge
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pyproj import Transformer
import pystac_client
import planetary_computer

from workshop_utils import RAW_DIR, PROCESSED_DIR
print(f'RAW_DIR: {RAW_DIR}')

---
## 1. Beck et al. (2023) — Köppen-Geiger maps and 1 km climate data

> Beck, H. E., et al. (2023). High-resolution (1 km) Köppen-Geiger maps

Already done!

---
## 2. GBIF species occurrence records

Two species, two purposes:
- **Philippine Eagle** (*Pithecophaga jefferyi*, GBIF taxonKey 2480381) — Day 2 biodiversity showcase.
  Even with few records, the map tells a powerful conservation story.
- **Philippine Flying Fox** (*Pteropus vampyrus*, GBIF taxonKey 5218644) — Day 3 SDM target.
  Widespread across the archipelago, with enough records for a meaningful model.

⚠️ Verify taxonKeys via `GET /v1/species/match?name=...` before use — GBIF backbone taxonomy
keys can go stale (the previous keys for both species silently returned 0 PH records).

No API key required. Pagination via `offset`.

In [ ]:
def download_gbif(taxon_key, country='PH', out_path=None):
    records, offset, limit = [], 0, 300
    base = 'https://api.gbif.org/v1/occurrence/search'
    while True:
        r = requests.get(base, params={
            'taxonKey': taxon_key, 'country': country,
            'hasCoordinate': 'true', 'hasGeospatialIssue': 'false',
            'limit': limit, 'offset': offset,
        })
        r.raise_for_status()
        data = r.json()
        records.extend(data['results'])
        print(f"  {len(records)}/{data['count']}", end='\r')
        if data['endOfRecords']:
            break
        offset += limit
    print()

    df = pd.DataFrame([
        {
            'species':  rec.get('species', ''),
            'lon':      rec.get('decimalLongitude'),
            'lat':      rec.get('decimalLatitude'),
            'year':     rec.get('year'),
            'basis':    rec.get('basisOfRecord', ''),
            'gbifID':   rec.get('gbifID'),
        }
        for rec in records if rec.get('decimalLongitude') is not None
    ])

    if out_path is not None:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_path, index=False)
        print(f'  saved {len(df)} records -> {out_path}')
    return df

In [ ]:
eagle = download_gbif(
    taxon_key=2480381,  # Pithecophaga jefferyi (verified via /v1/species/match; old key 2480965 was stale, returned 0 PH records)
    out_path=RAW_DIR / 'GBIF' / 'philippine_eagle.csv',
)
eagle.head()

In [ ]:
fox = download_gbif(
    taxon_key=5218644,  # Pteropus vampyrus (old key 2432859 was stale and returned 0 PH records)
    out_path=RAW_DIR / 'GBIF' / 'flying_fox.csv',
)
print(fox['year'].value_counts().sort_index().tail(15))
fox.head()

In [ ]:
# --- Clean + copy to PROCESSED_DIR/day_2 for student use ---
day2_dir = PROCESSED_DIR / "day_2"
day2_dir.mkdir(parents=True, exist_ok=True)

for label, fname in [("philippine_eagle", "philippine_eagle.csv"), ("flying_fox", "flying_fox.csv")]:
    df = pd.read_csv(RAW_DIR / "GBIF" / fname)
    df = (
        df.dropna(subset=["lon", "lat"])
          .drop_duplicates(subset=["gbifID"])
          .query("114 <= lon <= 127 and 4 <= lat <= 22")  # sanity-check: inside PH bounds
          .sort_values("year")
    )
    df.to_csv(day2_dir / f"gbif_{label}.csv", index=False)
    print(f"saved gbif_{label}.csv ({len(df)} rows)")

### 2b. GBIF mammal community dataset — urbanisation & terrestrial mammals in the Philippines

A much richer complement to the two single-species pulls above: a compiled GBIF occurrence
download of **all present-day terrestrial mammal records for the Philippines**
(class Mammalia, `COUNTRY=PH`, `OCCURRENCE_STATUS=present`), assembled from iNaturalist,
museum specimens, barcoding projects, etc. Backs the paper *"Making it in the city: Public
data reveal patterns and drivers of terrestrial mammal occurrence across urban Philippines."*

- 67,250 raw records → 289 species after cleaning
- Includes IUCN Red List category, island/province/municipality, and ~5,500 Negros-tagged
  records — good direct overlap with our Sentinel-2 / Hansen Negros data for Part IV
  (urban expansion vs. mammal occurrence)
- GBIF download DOI: [10.15468/dl.6ju8sc](https://doi.org/10.15468/dl.6ju8sc) — **cite this
  DOI (see `citations.txt` in the download) whenever the dataset is used**, since it is a
  compilation of many constituent datasets.

In [ ]:
import zipfile

GBIF_DOWNLOAD_KEY = "0003657-260226173443078"
GBIF_DOI_URL = f"https://api.gbif.org/v1/occurrence/download/request/{GBIF_DOWNLOAD_KEY}.zip"

mammals_dir = RAW_DIR / "GBIF" / "mammals_philippines"
mammals_dir.mkdir(parents=True, exist_ok=True)
zip_path = mammals_dir / f"{GBIF_DOWNLOAD_KEY}.zip"

if not zip_path.exists():
    print("downloading GBIF mammal occurrence download...")
    r = requests.get(GBIF_DOI_URL, timeout=120)
    r.raise_for_status()
    zip_path.write_bytes(r.content)
    print(f"saved {zip_path}  ({zip_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("skip download, zip exists")

with zipfile.ZipFile(zip_path) as z:
    for member in ["occurrence.txt", "citations.txt", "rights.txt"]:
        if not (mammals_dir / member).exists():
            z.extract(member, mammals_dir)
print("extracted occurrence.txt, citations.txt, rights.txt")

In [ ]:
# --- Process: trim to workshop-relevant columns, clean, save to PROCESSED_DIR/day_2 ---
KEEP_COLS = [
    "gbifID", "species", "genus", "family", "order", "class",
    "decimalLongitude", "decimalLatitude", "coordinateUncertaintyInMeters",
    "year", "month", "day", "basisOfRecord", "individualCount",
    "island", "islandGroup", "stateProvince", "county", "municipality", "locality",
    "iucnRedListCategory", "establishmentMeans", "institutionCode", "datasetKey",
]

mammals = pd.read_csv(mammals_dir / "occurrence.txt", sep="\t", usecols=KEEP_COLS, low_memory=False)
before = len(mammals)
mammals = mammals.dropna(subset=["decimalLongitude", "decimalLatitude", "species"])
mammals = mammals[mammals["class"] == "Mammalia"]
print(f"{before} -> {len(mammals)} rows after cleaning  ({mammals['species'].nunique()} species)")
print(mammals["iucnRedListCategory"].value_counts(dropna=False))

negros_mask = (mammals["county"].astype(str).str.contains("NEGROS", case=False, na=False) |
               mammals["stateProvince"].astype(str).str.contains("Negros", case=False, na=False))
print(f"Negros-tagged records: {negros_mask.sum()}")

out_csv = day2_dir / "gbif_mammals_philippines.csv"
mammals.to_csv(out_csv, index=False)
print(f"saved {out_csv}  ({out_csv.stat().st_size / 1e6:.1f} MB)")

---
## 3. Multispectral imagery for land-cover comparison — Negros

Two composites, built to actually be comparable:
- **Historical**: **Landsat 5 TM**, ~1990-1996
- **Recent**: **Sentinel-2 L2A**, ~2022-2024

That ~32-year gap (vs. a single 2018-2024 pull) gives students a genuinely dramatic, teachable
land-use-change signal instead of noise from a 6-year window.

⚠️ Two problems with grabbing "the lowest-cloud scene per tile" (the original approach):
1. **Clouds/shadows leave real gaps** even in the least-cloudy single scene available — there's no
   fallback data to fill them.
2. **Season contaminates the comparison.** Comparing scenes from different months of the year (e.g.
   Feb vs. April) confounds seasonal NDVI/greenness shifts with real land-cover change.

Fix for both: instead of one scene, pool **many** scenes over a **fixed season** across **several
years**, mask clouds/shadows per-pixel using each sensor's own QA band, and take the **per-pixel
temporal median**. Gaps in one scene get filled by another date; season is held constant; outlier
cloud/shadow pixels get median'd away.

We use the **Philippine dry season (Feb-Apr)** — confirmed empirically below to be far less cloudy
over Negros than the wet-season months (median cloud cover ~18% vs. ~53% for Jun-Aug).

Sources — both public STAC catalogs, no credentials needed:
- Sentinel-2 L2A: Element84 Earth Search (`sentinel-2-l2a`), using its `scl` (Scene Classification
  Layer) band for cloud/shadow masking
- Landsat 5 TM: Microsoft Planetary Computer (`landsat-c2-l2`), using its `qa_pixel` band (bit 6 =
  "clear") for masking; DN converted to physical surface reflectance via the official USGS scale/offset
  (×0.0000275, −0.2) — unlike Sentinel-2's pure multiplicative reflectance scale, Landsat Collection 2's
  additive offset means NDVI computed on raw DN would be wrong

Both are read via a **fast windowed + decimated native-resolution read** first (exploiting each COG's
built-in overviews), *then* reprojected onto one common ~100 m lat/lon grid so every scene — any tile,
any date, either sensor — stacks pixel-for-pixel before compositing. (Reprojecting directly against the
remote COG instead, e.g. via a naive `WarpedVRT`, turned out ~4x slower in testing — it bypasses the
overviews GDAL would otherwise use.)

Output:
- `data/raw/Sentinel2/negros_imagery.nc` — full composite, ~100 m
- `data/processed/day_2/negros_imagery.nc` — coarsened ~200 m, exercise-ready

Note: this is a **separate file** from `negros_sentinel2.nc` (the original single-scene 2018/2024
pull), which is left untouched — Part IV of `day2_solutions.ipynb` relies on exactly that file's
season-mismatch/cloud-gap flaws as a teaching point (a naive classifier overestimating real land-use
change). `negros_imagery.nc` is for Part II's plain eyeball comparison only.

In [ ]:
NEGROS_BBOX = [122.3, 9.8, 123.4, 11.0]   # W, S, E, N
TARGET_RES_M = 100
DEG_PER_M = 1 / 111_320   # approx meters-per-degree at the equator; fine for a teaching-grade grid

GRID_RES = TARGET_RES_M * DEG_PER_M
GRID_WIDTH = int(round((NEGROS_BBOX[2] - NEGROS_BBOX[0]) / GRID_RES))
GRID_HEIGHT = int(round((NEGROS_BBOX[3] - NEGROS_BBOX[1]) / GRID_RES))
GRID_TRANSFORM = rasterio.Affine(GRID_RES, 0, NEGROS_BBOX[0], 0, -GRID_RES, NEGROS_BBOX[3])
GRID_CRS = "EPSG:4326"
LON = NEGROS_BBOX[0] + GRID_RES * (np.arange(GRID_WIDTH) + 0.5)
LAT = NEGROS_BBOX[3] - GRID_RES * (np.arange(GRID_HEIGHT) + 0.5)
print(f"common grid: {GRID_WIDTH}x{GRID_HEIGHT} at ~{TARGET_RES_M}m")

# --- Is Jun-Aug (JJA, Northern-Hemisphere "summer") or Feb-Apr (FMA) actually less cloudy over Negros? ---
_catalog_check = pystac_client.Client.open("https://earth-search.aws.element84.com/v1")
for label, start, end in [("JJA (Jun-Aug)", "06-01", "08-31"), ("FMA (Feb-Apr)", "02-01", "04-30")]:
    clouds = []
    for year in range(2021, 2025):
        items = list(_catalog_check.search(
            collections=["sentinel-2-l2a"], bbox=NEGROS_BBOX,
            datetime=f"{year}-{start}/{year}-{end}",
            query={"eo:cloud_cover": {"lt": 90}}, max_items=300,
        ).items())
        clouds.extend(it.properties["eo:cloud_cover"] for it in items)
    print(f"  {label}: n={len(clouds)}, median cloud%={np.median(clouds):.1f}")
# -> FMA wins clearly: this is the Philippine dry season ("tag-init"), not JJA (SW monsoon / habagat)

# --- Pool scenes: Sentinel-2 L2A (recent) + Landsat 5 TM (historical), both Feb-Apr, several years ---
print("\nSearching Sentinel-2 L2A (recent, Feb-Apr 2022-2024, cloud<60%)...")
catalog = pystac_client.Client.open("https://earth-search.aws.element84.com/v1")
s2_items = []
for year in range(2022, 2025):
    search = catalog.search(collections=["sentinel-2-l2a"], bbox=NEGROS_BBOX,
                             datetime=f"{year}-02-01/{year}-04-30",
                             query={"eo:cloud_cover": {"lt": 60}}, max_items=300)
    s2_items.extend(search.items())
print(f"  {len(s2_items)} scenes pooled")

print("\nSearching Landsat 5 TM (historical, Feb-Apr 1990-1996, cloud<70%) via Planetary Computer...")
pc_catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)
l5_items = []
for year in range(1990, 1997):
    search = pc_catalog.search(collections=["landsat-c2-l2"], bbox=NEGROS_BBOX,
                                datetime=f"{year}-02-01/{year}-04-30",
                                query={"platform": {"in": ["landsat-5"]},
                                       "landsat:collection_category": {"in": ["T1"]},
                                       "eo:cloud_cover": {"lt": 70}},
                                max_items=100)
    l5_items.extend(search.items())
print(f"  {len(l5_items)} scenes pooled")

In [ ]:
def read_band_on_grid(href, resampling, target_res_m=TARGET_RES_M):
    """Fast windowed + decimated native-CRS read (uses each COG's built-in overviews),
    then a cheap in-memory reproject of the already-small result onto the common Negros grid."""
    with rasterio.open(href) as src:
        t = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        xmin, ymin = t.transform(NEGROS_BBOX[0], NEGROS_BBOX[1])
        xmax, ymax = t.transform(NEGROS_BBOX[2], NEGROS_BBOX[3])
        win = from_bounds(xmin, ymin, xmax, ymax, src.transform).intersection(
            rasterio.windows.Window(0, 0, src.width, src.height))
        if win.width <= 0 or win.height <= 0:
            return None   # this scene doesn't actually overlap Negros
        decim = max(1, round(target_res_m / src.res[0]))
        out_h = max(1, int(win.height / decim))
        out_w = max(1, int(win.width / decim))
        data = src.read(1, window=win, out_shape=(out_h, out_w), resampling=resampling)
        win_transform = src.window_transform(win)
        native_transform = win_transform * rasterio.Affine.scale(win.width / out_w, win.height / out_h)
        src_crs, src_nodata = src.crs, (src.nodata if src.nodata is not None else 0)

    dst = np.full((GRID_HEIGHT, GRID_WIDTH), np.nan, dtype="float32")
    reproject(source=data.astype("float32"), destination=dst,
              src_transform=native_transform, src_crs=src_crs, src_nodata=src_nodata,
              dst_transform=GRID_TRANSFORM, dst_crs=GRID_CRS, dst_nodata=np.nan,
              resampling=resampling)
    return dst


def composite_period(items, band_assets, qc_asset, is_clear_fn, scale=1.0, offset=0.0, label=""):
    """Per-pixel cloud/shadow-masked temporal median composite across all pooled scenes."""
    n = len(items)
    stacks = {b: np.full((n, GRID_HEIGHT, GRID_WIDTH), np.nan, dtype="float32") for b in band_assets}
    for i, it in enumerate(items):
        qc = read_band_on_grid(it.assets[qc_asset].href, Resampling.nearest)
        if qc is None:
            continue
        clear = is_clear_fn(np.nan_to_num(qc, nan=0).astype("uint16"))
        for band, asset_key in band_assets.items():
            data = read_band_on_grid(it.assets[asset_key].href, Resampling.average)
            if data is not None:
                stacks[band][i] = np.where(clear, data * scale + offset, np.nan)
        if (i + 1) % 10 == 0 or (i + 1) == n:
            print(f"  [{label}] {i + 1}/{n}")
    with np.errstate(all="ignore"):
        out = {b: np.nanmedian(s, axis=0) for b, s in stacks.items()}
    for b, arr in out.items():
        print(f"  [{label}] {b}: {np.isnan(arr).mean():.1%} NaN in composite")
    return out


# Sentinel-2 L2A: SCL classes 4/5/6 = vegetation / bare soil / water (drop cloud/shadow/cirrus/snow/defective)
s2_bands = {"red": "red", "green": "green", "blue": "blue", "nir": "nir"}
s2_clear = lambda scl: np.isin(scl, [4, 5, 6])
s2_composite = composite_period(s2_items, s2_bands, "scl", s2_clear, scale=1 / 10000, label="S2")

# Landsat 5 TM: QA_PIXEL bit 6 = "clear" (no cloud, dilated cloud, cirrus, or cloud shadow)
l5_bands = {"red": "red", "green": "green", "blue": "blue", "nir": "nir08"}
l5_clear = lambda qa: ((qa.astype(np.uint16) >> 6) & 1).astype(bool)
l5_composite = composite_period(l5_items, l5_bands, "qa_pixel", l5_clear, scale=0.0000275, offset=-0.2, label="L5")

In [ ]:
# Combine both periods into a single NetCDF with a 'time' dimension
def to_uint8_stretch(band, vmax=0.3):
    """simple linear reflectance stretch (0-vmax -> 0-255) for true-colour display"""
    return np.clip(band, 0, vmax) / vmax * 255


def build_period_vars(comp):
    out = {b: (("latitude", "longitude"), comp[b]) for b in ["red", "green", "blue", "nir"]}
    out["visual_r"] = (("latitude", "longitude"), to_uint8_stretch(comp["red"]))
    out["visual_g"] = (("latitude", "longitude"), to_uint8_stretch(comp["green"]))
    out["visual_b"] = (("latitude", "longitude"), to_uint8_stretch(comp["blue"]))
    return out


def median_year(items):
    return int(np.median([pd.Timestamp(it.properties["datetime"][:10]).year for it in items]))


l5_time = pd.Timestamp(f"{median_year(l5_items)}-03-15")   # nominal mid-dry-season representative date
s2_time = pd.Timestamp(f"{median_year(s2_items)}-03-15")

ds_hist = xr.Dataset(build_period_vars(l5_composite), coords={"latitude": LAT, "longitude": LON}).expand_dims(time=[l5_time])
ds_recent = xr.Dataset(build_period_vars(s2_composite), coords={"latitude": LAT, "longitude": LON}).expand_dims(time=[s2_time])

combined = xr.concat([ds_hist, ds_recent], dim="time")
for v in combined.data_vars:
    combined[v] = combined[v].astype("float32")

combined.attrs["description"] = (
    "Cloud-free, season-matched composite imagery over Negros: Landsat 5 TM (historical) and "
    "Sentinel-2 L2A (recent), each median-composited from many Feb-Apr (Philippine dry season) "
    "scenes pooled across several years, to suppress cloud/shadow gaps and remove wet/dry-season "
    "NDVI bias from the historical-vs-recent comparison."
)
combined.attrs["historical_source"] = f"Landsat 5 TM, {len(l5_items)} scenes, Feb-Apr 1990-1996, cloud<70%"
combined.attrs["recent_source"] = f"Sentinel-2 L2A, {len(s2_items)} scenes, Feb-Apr 2022-2024, cloud<60%"
combined.attrs["bands_note"] = (
    "red/green/blue/nir are physical surface reflectance (0-1 float; NDVI=(nir-red)/(nir+red) works "
    "directly on these); NaN means no clear observation existed anywhere in the pooled scenes for that "
    "pixel; visual_r/g/b are a simple 0-0.3 reflectance stretch to 0-255 for true-colour display"
)
combined.attrs["time_note"] = (
    "time coordinates are nominal representative dates (median year of the pooled scenes, mid-dry-season) "
    "-- each time slice is a multi-year median composite, not a single acquisition"
)

out_nc = RAW_DIR / "Sentinel2" / "negros_imagery.nc"
out_nc.parent.mkdir(parents=True, exist_ok=True)
combined.to_netcdf(out_nc)
print(f"Saved: {out_nc}")
print(combined)

In [ ]:
# --- Exercise-ready version: coarsen ~2x (100m -> ~200m) to keep the file small ---
day2_dir = PROCESSED_DIR / "day_2"
day2_dir.mkdir(parents=True, exist_ok=True)

ds_coarse = combined.coarsen(latitude=2, longitude=2, boundary="trim").mean()
for v in ds_coarse.data_vars:
    ds_coarse[v] = ds_coarse[v].astype("float32")
ds_coarse.attrs = combined.attrs
ds_coarse.attrs["resolution_note"] = "coarsened ~2x from the ~100m native composite for a lighter workshop file (~200m)"

out_path = day2_dir / "negros_imagery.nc"
ds_coarse.to_netcdf(out_path)
print(f"Saved {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")

---
## 4. Hansen Global Forest Change — Negros

Hansen et al. (2013) provide annual forest cover and loss data from 2000 to present
at 30 m resolution as public COGs on Google Cloud Storage.

Layers:
- `treecover2000` — canopy cover in year 2000 (%)
- `lossyear` — year of first forest loss (1=2001 … 23=2023, 0=no loss)

Negros (9–11°N) straddles two 10°×10° tiles (`20N_120E` and `10N_120E`).
We use rasterio windowed reads on the remote COGs — only the Negros-sized patch
is transferred, not the full global tile.

In [ ]:
HANSEN_BASE   = 'https://storage.googleapis.com/earthenginepartners-hansen/GFC-2023-v1.11'
HANSEN_LAYERS = ['treecover2000', 'lossyear']
HANSEN_TILES  = ['20N_120E', '10N_120E']   # northern tile first (higher latitude)
NEGROS_FULL   = (121.5, 8.8, 124.5, 11.5)  # W, S, E, N — slightly wider than NEGROS_BBOX

out_dir = RAW_DIR / 'Hansen'
out_dir.mkdir(parents=True, exist_ok=True)

for layer in HANSEN_LAYERS:
    out_path = out_dir / f'hansen_{layer}_negros.tif'
    if out_path.exists():
        print(f'skip {out_path.name}')
        continue

    print(f'{layer}:')
    mem_files = []
    for tile in HANSEN_TILES:
        url = f'{HANSEN_BASE}/Hansen_GFC-2023-v1.11_{layer}_{tile}.tif'
        print(f'  reading {tile} ...', end='', flush=True)
        try:
            with rasterio.open(url) as src:
                win = from_bounds(*NEGROS_FULL, src.transform)
                row0 = max(0, int(win.row_off))
                col0 = max(0, int(win.col_off))
                row1 = min(src.height, int(win.row_off + win.height))
                col1 = min(src.width,  int(win.col_off + win.width))
                if row1 <= row0 or col1 <= col0:
                    print(' no overlap — skip')
                    continue
                win_c = rasterio.windows.Window(col0, row0, col1 - col0, row1 - row0)
                data = src.read(1, window=win_c)
                transform = src.window_transform(win_c)
                profile = src.profile.copy()
                profile.update(
                    width=data.shape[1], height=data.shape[0],
                    transform=transform, count=1,
                )
            mem = MemoryFile()
            with mem.open(**profile) as mds:
                mds.write(data, 1)
            mem_files.append(mem)
            print(f' {data.shape}')
        except Exception as e:
            print(f' ERROR: {e}')

    if not mem_files:
        print(f'  no data for {layer}')
        continue

    opened = [m.open() for m in mem_files]
    mosaic, out_transform = rio_merge(opened)
    profile_out = opened[0].profile.copy()
    profile_out.update(
        height=mosaic.shape[1], width=mosaic.shape[2],
        transform=out_transform, count=1,
    )
    for d in opened:
        d.close()
    for m in mem_files:
        m.close()

    with rasterio.open(out_path, 'w', **profile_out) as dst:
        dst.write(mosaic[0], 1)

    print(f'  -> saved {out_path.name}  {mosaic.shape}')

print('\nHansen download complete.')

In [ ]:
# --- Build exercise-ready netCDF: crop tight to NEGROS_BBOX, reproject to plain lat/lon, compact dtypes ---
day2_dir = PROCESSED_DIR / "day_2"
day2_dir.mkdir(parents=True, exist_ok=True)

layer_arrays = {}
for layer in HANSEN_LAYERS:
    da = rioxarray.open_rasterio(out_dir / f"hansen_{layer}_negros.tif").squeeze("band", drop=True)
    layer_arrays[layer] = da

hansen_ds = xr.Dataset(layer_arrays).rename({"x": "longitude", "y": "latitude"})
hansen_ds = hansen_ds.sel(longitude=slice(NEGROS_BBOX[0], NEGROS_BBOX[2]),
                           latitude=slice(NEGROS_BBOX[3], NEGROS_BBOX[1]))
hansen_ds["treecover2000"] = hansen_ds["treecover2000"].astype("uint8")
hansen_ds["lossyear"] = hansen_ds["lossyear"].astype("uint8")
hansen_ds.attrs["description"] = "Hansen Global Forest Change v1.11 (2023), cropped to Negros, native ~30m resolution"
hansen_ds.attrs["source"] = "Hansen et al. (2013), https://storage.googleapis.com/earthenginepartners-hansen/"
hansen_ds.attrs["lossyear_note"] = "year of first forest loss: 1=2001 ... 23=2023, 0=no loss detected"

out_nc = day2_dir / "negros_forest_change.nc"
hansen_ds.to_netcdf(out_nc)
print(f"Saved {out_nc}  ({out_nc.stat().st_size / 1e6:.1f} MB)")
print(hansen_ds)